In [0]:
%pip install kagglehub

In [0]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("frankossai/apple-stock-aapl-historical-financial-news-data")

print("Path to dataset files:", path)

In [0]:
import os

path = "/home/spark-91796150-64e5-4dac-a1f8-ac/.cache/kagglehub/datasets/frankossai/apple-stock-aapl-historical-financial-news-data/versions/1"

files = os.listdir(path)
print(files)

In [0]:
import pandas as pd

file_path = f"{path}/apple_news_data.csv"

historical_raw_df = pd.read_csv(file_path)

print("Columns:", historical_raw_df.columns.tolist())
print("Rows:", len(historical_raw_df))
historical_raw_df.head()

In [0]:
import pandas as pd
import numpy as np

df = historical_raw_df.copy()

# Standardize types
df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)

# Keep rows where symbols mention AAPL
df["symbols"] = df["symbols"].fillna("").astype(str)
df = df[df["symbols"].str.contains("AAPL", case=False, na=False)].copy()

# Build normalized columns
df["published_at"] = df["date"]
df["Date"] = df["published_at"].dt.normalize()
df["Ticker"] = "AAPL"

df["headline"] = df["title"].fillna("").astype(str)
df["summary"] = df["content"].fillna("").astype(str)
df["source"] = "kaggle_aapl_dataset"
df["url"] = df["link"].fillna("").astype(str)

# Use existing sentiment columns
df["headline_sentiment"] = df["sentiment_polarity"].fillna(0.0).astype(float)
df["summary_sentiment"] = df["sentiment_polarity"].fillna(0.0).astype(float)

# Combined sentiment
df["combined_sentiment"] = (
    0.7 * df["headline_sentiment"] +
    0.3 * df["summary_sentiment"]
)

# Keep only the columns we want downstream
aapl_news_hist_df = df[[
    "published_at",
    "Date",
    "Ticker",
    "headline",
    "summary",
    "source",
    "url",
    "headline_sentiment",
    "summary_sentiment",
    "combined_sentiment",
    "sentiment_neg",
    "sentiment_neu",
    "sentiment_pos",
    "symbols",
    "tags"
]].copy()

# Drop obvious junk
aapl_news_hist_df = aapl_news_hist_df.dropna(subset=["Date"])
aapl_news_hist_df = aapl_news_hist_df.drop_duplicates(
    subset=["Ticker", "published_at", "headline"]
).reset_index(drop=True)

print("Rows after AAPL filter:", len(aapl_news_hist_df))
aapl_news_hist_df.head(10)

In [0]:
print(aapl_news_hist_df.columns.tolist())
print(aapl_news_hist_df[[
    "headline_sentiment",
    "summary_sentiment",
    "combined_sentiment",
    "sentiment_neg",
    "sentiment_neu",
    "sentiment_pos"
]].describe())

print(aapl_news_hist_df["Date"].min(), aapl_news_hist_df["Date"].max())

In [0]:
spark.createDataFrame(aapl_news_hist_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_universe_news_raw")

In [0]:
print("Rows after AAPL filter:", len(aapl_news_hist_df))
print(aapl_news_hist_df["Date"].min(), aapl_news_hist_df["Date"].max())
print(aapl_news_hist_df.columns.tolist())